In [0]:
%pip install databricks_langchain==0.9.0 -q
%pip install databricks-vectorsearch -q
%pip install mlflow-skinny[databricks] -q
%pip install langchain==1.0.0 -q
%pip install langgraph==1.0.10 -q

dbutils.library.restartPython()

In [0]:
import json
import mlflow
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.state import CompiledStateGraph

mlflow.langchain.autolog()

LLM_ENDPOINT_NAME = "databricks-gpt-oss-120b"
INDEX_NAME = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_chunked_index"
SYSTEM_PROMPT = """You are expert on Sephora's pricing strategy. Respond in a clear, professional, and factual tone
    appropriate for engineers and technical staff. Use only verified information from the provided context, include 
    reference to every information you provide. Do not hallucinate information. If you are unsure about the answer,
    say "I don't know"."""
DESCRIPTION = "Sephora pricing strategy"
MODEL_NAME = "sephora_knowledge_assistant"
UC_MODEL_NAME = "workspace.sephora_products_and_skincare_reviews.sephora_knowledge_assistant"

In [0]:
import yaml

def create_agent_config(llm_endpoint_name: str, index_name: str, num_results: int = 3) -> dict:
    config = {
        "api_version": "1",
        "llm_endpoint_name": llm_endpoint_name,
        'vector_search':{
            "index_name": index_name,
            "num_results": num_results,
        },
        "system_prompt": SYSTEM_PROMPT,
        "description": DESCRIPTION,
    }
    return config


agent_config = create_agent_config(LLM_ENDPOINT_NAME, INDEX_NAME, num_results=3)
config_yaml = yaml.safe_dump(agent_config, sort_keys=False)
config_yaml_filepath = 'agent_config_pricing.yaml'

with open(config_yaml_filepath, 'w', encoding='utf-8') as file:
    file.write(config_yaml)

print(config_yaml)

In [0]:
import mlflow
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from importlib.metadata import version as get_version


tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "sephora_knowledge_base"
}

input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is Sephora's pricing strategy",
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name=MODEL_NAME,
        python_model="agent_pricing.py",
        code_paths=[config_yaml_filepath, "agent.py"],
        input_example=input_example,
        pip_requirements=[
            f"mlflow=={get_version('mlflow')}",
            f"langchain=={get_version('langchain')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
        ],
        resources=[
            DatabricksVectorSearchIndex(index_name=INDEX_NAME),
            DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
        ],
        registered_model_name=UC_MODEL_NAME
    )

    model_uri = logged_agent_info.model_uri

print(f"Model URI: {model_uri}")

In [0]:
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = "workspace.sephora_products_and_skincare_reviews.sephora_knowledge_assistant"

uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri,
    name=UC_MODEL_NAME
)

print(uc_registered_model_info.version)


In [0]:
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=pyfunc_model.input_example,
    env_manager="uv"
)

**Sephora’s pricing strategy is a data‑driven, AI‑enabled, market‑competitive approach that blends profitability targets with real‑time competitor intelligence.**  

The key components, as described in the case‑study material, are:

| Component | Description (source) |
|-----------|----------------------|
| **Centralised competitive‑data platform** – Sephora adopted Competera’s Competitive Data solution to collect **shelf‑price, promotion and coupon information** from rivals across 13 countries. The platform delivers **millions of data points daily** (e.g., >5.9 M data points per day, 10.9 M monthly from direct competitors) and stores historical price context for ML‑ready use. (Document 0, 2) |
| **Real‑time & historical analytics** – Near‑real‑time feeds (processed before 9 AM) are combined with long‑term price history (stock periods, delivery terms, brand availability) to support **advanced pricing algorithms**. (Document 1) |
| **ML‑based price optimisation** – The high‑quality, **98 % match quality** and **99 % data accuracy** enable sophisticated pricing models that can **dynamically adjust prices** while respecting brand‑level and regional rules. (Document 1) |
| **Localized pricing strategies** – By standard

In [0]:
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

result = mlflow.models.predict(
    model_uri=model_uri,
    input_data={
        "input": [
            {
                "role": "user",
                "content": "What is metioned regarding pricing strategy in Poland",
            }
        ]
    },
    env_manager="uv"
)

**Poland’s role in Sephora’s pricing‑strategy rollout**

- **Proof‑of‑Concept (POC) pilot** – Poland (together with Germany) was selected for a one‑month pilot of Competera’s Competitive Data solution. The pilot was intended to test a “significant pricing transformation” by delivering ML‑ready, real‑time price information at the SKU level【Document id 2.0】.  

- **Scope and coverage** – The POC covered **1,000 SKUs per market**, with daily delivery of both **shelf‑price